# ADD 2026 Track 2 — MOSS-Audio 4B LoRA

실행 순서: **B. 8-sample overfit → C. Rank-16 100-step profile → D. Rank-16 1 epoch → Rank-32 1 epoch → 승자 3 epochs**. MOSS-Audio 공식 런타임에 맞춰 Transformers 4.57.1을 사용하며 `bert-score`는 평가 직전에 없을 때 설치합니다.

In [ ]:
from pathlib import Path

PROJECT_REPO = "https://github.com/youhan200203/ADD2026-track2-baseline.git"
PROJECT_DIR = Path("/content/ADD2026-track2-baseline")
MOSS_REPO = Path("/content/MOSS-Audio")
MODEL_ID = "OpenMOSS-Team/MOSS-Audio-4B-Instruct"
MODEL_DIR = Path("/content/MOSS-Audio-4B-Instruct")
DATASET_FILE_ID = "1QQIEf5lE5I5dIpKrnxGW8noj71zxZmd8"
DATA_ARCHIVE = Path("/content/track2_training_datasets.zip")
DATA_ROOT = Path("/content/track2_training_datasets")
MOSS_DATA_DIR = PROJECT_DIR / "data/moss"
DRIVE_ROOT = Path("/content/drive/MyDrive/ADD2026/moss_audio_4b")
OVERFIT_DIR = DRIVE_ROOT / "overfit_r16"
PROFILE_DIR = DRIVE_ROOT / "profile_r16_dynamic_no_gc_100steps"
R16_DIR = DRIVE_ROOT / "r16_epoch1"
R32_DIR = DRIVE_ROOT / "r32_epoch1"
MAX_LEN_CAP = 4096
MAX_NEW_TOKENS = 2048
PER_DEVICE_TRAIN_BATCH_SIZE = 1
GRAD_ACCUMULATION = 8
LEARNING_RATE = 2e-4
OVERFIT_STEPS = 100
PROFILE_STEPS = 100
SEED = 42

In [ ]:
import importlib.util, shutil, sys
from importlib.metadata import version as package_version
required_modules = ["accelerate", "gdown", "huggingface_hub", "librosa", "peft", "soundfile", "torch", "transformers"]
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
missing_commands = [name for name in ("git",) if shutil.which(name) is None]
if missing_modules or missing_commands:
    raise RuntimeError({"missing_modules": missing_modules, "missing_commands": missing_commands})
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required")
print({"python": sys.version.split()[0], "torch": torch.__version__, "transformers_before_setup": package_version("transformers"), "gpu": torch.cuda.get_device_name(0)})

In [ ]:
!python -m pip uninstall -y torchao

In [ ]:
!python -m pip install -q transformers==4.57.1
!python -c "import transformers; assert transformers.__version__ == '4.57.1'; print('transformers', transformers.__version__)"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

## 코드, 데이터, 모델 준비

In [ ]:
if PROJECT_DIR.is_dir():
    print(f"[reuse] {PROJECT_DIR}")
else:
    !git clone {PROJECT_REPO} {PROJECT_DIR}
if MOSS_REPO.is_dir():
    print(f"[reuse] {MOSS_REPO}")
else:
    !git clone https://github.com/OpenMOSS/MOSS-Audio.git {MOSS_REPO}
for required in ("prepare_moss_data.py", "moss_train.py", "moss_infer.py"):
    if not (PROJECT_DIR / required).is_file():
        raise FileNotFoundError(PROJECT_DIR / required)
if not (MOSS_REPO / "finetune/finetune.py").is_file():
    raise FileNotFoundError(MOSS_REPO / "finetune/finetune.py")

In [ ]:
import gdown, os, zipfile
if not all((DATA_ROOT / name).is_dir() for name in ("train", "dev", "labels")):
    if not DATA_ARCHIVE.is_file():
        gdown.download(id=DATASET_FILE_ID, output=DATA_ARCHIVE.as_posix(), quiet=False)
    if not zipfile.is_zipfile(DATA_ARCHIVE):
        raise RuntimeError(f"Downloaded dataset is not a ZIP: {DATA_ARCHIVE}")
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:
        archive.extractall("/content")
if not all((DATA_ROOT / name).is_dir() for name in ("train", "dev", "labels")):
    raise RuntimeError(f"Expected train, dev, labels under {DATA_ROOT}")
project_data = PROJECT_DIR / "data"
project_data.mkdir(exist_ok=True)
for name in ("train", "dev", "labels"):
    target, source = project_data / name, DATA_ROOT / name
    if target.is_symlink():
        if target.resolve() != source.resolve():
            raise RuntimeError(f"Unexpected symlink: {target}")
    elif target.exists():
        raise RuntimeError(f"Refusing to replace: {target}")
    else:
        os.symlink(source, target, target_is_directory=True)
print({name: len(list((DATA_ROOT / name).glob("*.wav"))) for name in ("train", "dev")})

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id=MODEL_ID, local_dir=MODEL_DIR)
print(f"[ready] {MODEL_DIR}")

In [ ]:
os.chdir(PROJECT_DIR)
!python -u prepare_moss_data.py --data-root data --output-dir {MOSS_DATA_DIR} --overfit-per-class 4
!python -u prepare_data.py

## 실제 sequence 길이 분포

Train과 Dev의 오디오·prompt·정답 길이를 계산하고 최댓값을 128 단위로 올림합니다. 4,096을 넘는 경우만 상한에서 잘립니다.

In [ ]:
import json, math, numpy as np, soundfile as sf
from transformers import AutoTokenizer
def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
def sequence_lengths(path):
    lengths = []
    for row in read_jsonl(path):
        audio, prompt, answer = [part["content"] for part in row["conversation"]]
        info = sf.info(audio)
        audio_tokens = math.ceil((info.frames / info.samplerate) * 12.5)
        text_tokens = len(tokenizer.encode(prompt, add_special_tokens=False)) + len(tokenizer.encode(answer, add_special_tokens=False))
        lengths.append(23 + audio_tokens + text_tokens)
    return lengths
train_lengths = sequence_lengths(MOSS_DATA_DIR / "train.jsonl")
dev_lengths = sequence_lengths(MOSS_DATA_DIR / "dev.jsonl")
all_lengths = np.asarray(train_lengths + dev_lengths)
length_report = {name: float(np.percentile(all_lengths, value)) for name, value in {"p50": 50, "p95": 95, "p99": 99, "p99_5": 99.5}.items()}
length_report["max"] = int(all_lengths.max())
MAX_LEN = min(MAX_LEN_CAP, max(512, math.ceil(length_report["max"] / 128) * 128))
length_report["selected_max_len"] = MAX_LEN
length_report["truncated_samples"] = int((all_lengths > MAX_LEN).sum())
(DRIVE_ROOT / "sequence_length_report.json").write_text(json.dumps(length_report, indent=2), encoding="utf-8")
print(json.dumps(length_report, indent=2))
del tokenizer

## B. Overfit-one-batch

고정 Real 4개 + Fake 4개를 Rank 16으로 100 update steps 반복합니다. LoRA gradient, loss 하락, adapter 저장을 확인합니다.

In [ ]:
OVERFIT_MANIFEST = MOSS_DATA_DIR / "overfit_8.jsonl"

In [ ]:
OVERFIT_PROFILE = OVERFIT_DIR / "profile.json"
if (OVERFIT_DIR / "adapter_config.json").is_file() and OVERFIT_PROFILE.is_file():
    print(f"[skip] completed: {OVERFIT_DIR}")
else:
    !PYTHONPATH={MOSS_REPO} PYTHONUNBUFFERED=1 python -u moss_train.py \
      --moss-repo {MOSS_REPO} --profile-json {OVERFIT_PROFILE} --resume-from-checkpoint auto --require-lora-gradient \
      --model_dir {MODEL_DIR} --data_path {OVERFIT_MANIFEST} --output_dir {OVERFIT_DIR} \
      --use_lora --lora_rank 16 --lora_alpha 32 --attn_implementation sdpa \
      --bf16 True --tf32 True --gradient_checkpointing True --per_device_train_batch_size 1 \
      --gradient_accumulation_steps 1 --learning_rate 0.001 --max_steps {OVERFIT_STEPS} \
      --lr_scheduler_type cosine --warmup_steps 4 --logging_steps 1 --logging_first_step True \
      --save_strategy steps --save_steps 50 --save_total_limit 1 --report_to none \
      --remove_unused_columns False --dataloader_num_workers 2 --max_len {MAX_LEN} --seed {SEED}

In [ ]:
overfit_profile = json.loads(OVERFIT_PROFILE.read_text(encoding="utf-8"))
losses = [row["loss"] for row in overfit_profile["loss_history"]]
window = min(10, max(1, len(losses) // 2))
first_loss, last_loss = sum(losses[:window]) / window, sum(losses[-window:]) / window
if not last_loss < first_loss:
    raise RuntimeError(f"Overfit loss did not decrease: {first_loss} -> {last_loss}")
if overfit_profile["max_lora_gradient_norm"] <= 0:
    raise RuntimeError("LoRA gradient is zero")
if not (OVERFIT_DIR / "adapter_config.json").is_file():
    raise FileNotFoundError(OVERFIT_DIR / "adapter_config.json")
print({"first_loss": first_loss, "last_loss": last_loss, "max_lora_gradient_norm": overfit_profile["max_lora_gradient_norm"], "adapter_size_bytes": overfit_profile["adapter_size_bytes"]})

## C. Rank-16 100-step profile

gradient checkpointing을 끈 새 100-step run으로 peak VRAM, step time, samples/s, loss, 예상 1/3 epoch 시간, checkpoint 크기를 기록합니다. L4 OOM이면 본 학습은 gradient checkpointing을 켠 기존 설정을 유지합니다. QLoRA로 자동 변경하지 않습니다.

In [ ]:
PROFILE_JSON = PROFILE_DIR / "profile.json"
if (PROFILE_DIR / "adapter_config.json").is_file() and PROFILE_JSON.is_file():
    print(f"[skip] completed: {PROFILE_DIR}")
else:
    !PYTHONPATH={MOSS_REPO} PYTHONUNBUFFERED=1 python -u moss_train.py \
      --moss-repo {MOSS_REPO} --profile-json {PROFILE_JSON} --resume-from-checkpoint auto --require-lora-gradient \
      --model_dir {MODEL_DIR} --data_path {MOSS_DATA_DIR}/train.jsonl --output_dir {PROFILE_DIR} \
      --use_lora --lora_rank 16 --lora_alpha 32 --attn_implementation sdpa --bf16 True --tf32 True \
      --gradient_checkpointing False --per_device_train_batch_size {PER_DEVICE_TRAIN_BATCH_SIZE} --gradient_accumulation_steps {GRAD_ACCUMULATION} \
      --learning_rate {LEARNING_RATE} --max_steps {PROFILE_STEPS} --lr_scheduler_type cosine --warmup_steps 3 \
      --logging_steps 5 --logging_first_step True --save_strategy steps --save_steps 50 --save_total_limit 1 \
      --report_to none --remove_unused_columns False --dataloader_num_workers 2 --max_len {MAX_LEN} --seed {SEED}

In [ ]:
profile = json.loads(PROFILE_JSON.read_text(encoding="utf-8"))
profile_fields = ["peak_vram_allocated_gib", "peak_vram_reserved_gib", "step_time_seconds", "train_samples_per_second", "train_loss", "estimated_one_epoch_seconds", "estimated_three_epoch_seconds", "latest_checkpoint_size_bytes", "dynamic_padding_validation"]
padding_check = profile.get("dynamic_padding_validation") or {}
if profile["global_step"] != PROFILE_STEPS or profile["max_lora_gradient_norm"] <= 0 or profile["gradient_checkpointing"] or not padding_check.get("performed") or padding_check.get("trimmed_length", 0) >= padding_check.get("padded_length", 0):
    raise RuntimeError("100-step profile validation failed")
print(json.dumps({key: profile[key] for key in profile_fields}, indent=2))

## D. Rank-16 전체 Train 1 epoch

Train 8,600개만 학습하고 학습 중 Dev loss 평가는 하지 않습니다. Dev 2,000개는 뒤의 생성 기반 공식 평가에만 사용합니다. 200 update steps마다 Drive checkpoint를 저장하며 재실행하면 최신 checkpoint에서 자동 재개합니다.

In [ ]:
TRAIN_ROWS, DEV_ROWS = len(read_jsonl(MOSS_DATA_DIR / "train.jsonl")), len(read_jsonl(MOSS_DATA_DIR / "dev.jsonl"))
if (TRAIN_ROWS, DEV_ROWS) != (8600, 2000):
    raise RuntimeError(f"Expected 8600/2000, got {TRAIN_ROWS}/{DEV_ROWS}")
UPDATES_PER_EPOCH = math.ceil(TRAIN_ROWS / (PER_DEVICE_TRAIN_BATCH_SIZE * GRAD_ACCUMULATION))
WARMUP_1E = math.ceil(UPDATES_PER_EPOCH * 0.03)
WARMUP_3E = math.ceil(UPDATES_PER_EPOCH * 3 * 0.03)
print({"updates_per_epoch": UPDATES_PER_EPOCH, "warmup_1e": WARMUP_1E, "warmup_3e": WARMUP_3E})

In [ ]:
R16_PROFILE = R16_DIR / "profile.json"
if (R16_DIR / "adapter_config.json").is_file() and R16_PROFILE.is_file():
    print(f"[skip] completed: {R16_DIR}")
else:
    !PYTHONPATH={MOSS_REPO} PYTHONUNBUFFERED=1 python -u moss_train.py \
      --moss-repo {MOSS_REPO} --profile-json {R16_PROFILE} --resume-from-checkpoint auto --require-lora-gradient \
      --model_dir {MODEL_DIR} --data_path {MOSS_DATA_DIR}/train.jsonl \
      --output_dir {R16_DIR} --use_lora --lora_rank 16 --lora_alpha 32 --attn_implementation sdpa \
      --bf16 True --tf32 True --gradient_checkpointing True --per_device_train_batch_size {PER_DEVICE_TRAIN_BATCH_SIZE} \
      --gradient_accumulation_steps {GRAD_ACCUMULATION} --learning_rate {LEARNING_RATE} --num_train_epochs 1 \
      --lr_scheduler_type cosine --warmup_steps {WARMUP_1E} --logging_steps 10 --logging_first_step True \
      --save_strategy steps --save_steps 200 --save_total_limit 2 --report_to none \
      --remove_unused_columns False --dataloader_num_workers 2 --max_len {MAX_LEN} --seed {SEED}

## BERTScore 준비 및 Rank-16 Dev 생성 평가

In [ ]:
if importlib.util.find_spec("bert_score") is None:
    !python -m pip install -q bert-score
if importlib.util.find_spec("bert_score") is None:
    raise RuntimeError("bert-score installation failed")
print("[ready] bert_score")

In [ ]:
R16_PRED, R16_EVAL = DRIVE_ROOT / "r16_epoch1_dev.jsonl", DRIVE_ROOT / "r16_epoch1_eval.json"
!python -u moss_infer.py --moss-repo {MOSS_REPO} --model-dir {MODEL_DIR} --adapter-dir {R16_DIR} --audio-dir data/dev --output {R16_PRED} --max-new-tokens {MAX_NEW_TOKENS} --resume
!python -u eval_add2026.py {R16_PRED} --subset data/swift/dev.jsonl --out-json {R16_EVAL} --bert-batch-size 32 --bert-device cuda:0

## Rank-32 전체 Train 1 epoch — 나머지 조건은 Rank 16과 동일

In [ ]:
R32_PROFILE = R32_DIR / "profile.json"
if (R32_DIR / "adapter_config.json").is_file() and R32_PROFILE.is_file():
    print(f"[skip] completed: {R32_DIR}")
else:
    !PYTHONPATH={MOSS_REPO} PYTHONUNBUFFERED=1 python -u moss_train.py \
      --moss-repo {MOSS_REPO} --profile-json {R32_PROFILE} --resume-from-checkpoint auto --require-lora-gradient \
      --model_dir {MODEL_DIR} --data_path {MOSS_DATA_DIR}/train.jsonl \
      --output_dir {R32_DIR} --use_lora --lora_rank 32 --lora_alpha 64 --attn_implementation sdpa \
      --bf16 True --tf32 True --gradient_checkpointing True --per_device_train_batch_size {PER_DEVICE_TRAIN_BATCH_SIZE} \
      --gradient_accumulation_steps {GRAD_ACCUMULATION} --learning_rate {LEARNING_RATE} --num_train_epochs 1 \
      --lr_scheduler_type cosine --warmup_steps {WARMUP_1E} --logging_steps 10 --logging_first_step True \
      --save_strategy steps --save_steps 200 --save_total_limit 2 --report_to none \
      --remove_unused_columns False --dataloader_num_workers 2 --max_len {MAX_LEN} --seed {SEED}

In [ ]:
R32_PRED, R32_EVAL = DRIVE_ROOT / "r32_epoch1_dev.jsonl", DRIVE_ROOT / "r32_epoch1_eval.json"
!python -u moss_infer.py --moss-repo {MOSS_REPO} --model-dir {MODEL_DIR} --adapter-dir {R32_DIR} --audio-dir data/dev --output {R32_PRED} --max-new-tokens {MAX_NEW_TOKENS} --resume
!python -u eval_add2026.py {R32_PRED} --subset data/swift/dev.jsonl --out-json {R32_EVAL} --bert-batch-size 32 --bert-device cuda:0

## 1-epoch 승자 선택과 3-epoch 본 실험

In [ ]:
def compact_result(rank, path):
    result = json.loads(path.read_text(encoding="utf-8")); real, fake = result["real"], result["fake"]
    total = real["total"] + fake["total"]
    mean_bert = (real["mean_bertscore_f1_all"] * real["total"] + fake["mean_bertscore_f1_all"] * fake["total"]) / total
    return {"rank": rank, "balanced_accuracy": result["balanced_accuracy"], "mean_bertscore_f1": mean_bert, "final_score_100": result["final_score_100"]}
comparison = [compact_result(16, R16_EVAL), compact_result(32, R32_EVAL)]
comparison.sort(key=lambda row: (-row["final_score_100"], row["rank"]))
WINNER_RANK = comparison[0]["rank"]; WINNER_ALPHA = 2 * WINNER_RANK
WINNER_DIR = DRIVE_ROOT / f"r{WINNER_RANK}_epoch3"
WINNER_PROFILE = WINNER_DIR / "profile.json"
WINNER_PRED = DRIVE_ROOT / f"r{WINNER_RANK}_epoch3_dev.jsonl"
WINNER_EVAL = DRIVE_ROOT / f"r{WINNER_RANK}_epoch3_eval.json"
print(json.dumps({"experiments": comparison, "winner_rank": WINNER_RANK}, indent=2))

1-epoch adapter를 이어 붙이지 않고 베이스 4B에서 승자 rank로 정확히 3 epochs를 새로 학습합니다. 중단 시 이 run의 최신 Drive checkpoint에서 재개합니다.

In [ ]:
if (WINNER_DIR / "adapter_config.json").is_file() and WINNER_PROFILE.is_file():
    print(f"[skip] completed: {WINNER_DIR}")
else:
    !PYTHONPATH={MOSS_REPO} PYTHONUNBUFFERED=1 python -u moss_train.py \
      --moss-repo {MOSS_REPO} --profile-json {WINNER_PROFILE} --resume-from-checkpoint auto --require-lora-gradient \
      --model_dir {MODEL_DIR} --data_path {MOSS_DATA_DIR}/train.jsonl \
      --output_dir {WINNER_DIR} --use_lora --lora_rank {WINNER_RANK} --lora_alpha {WINNER_ALPHA} --attn_implementation sdpa \
      --bf16 True --tf32 True --gradient_checkpointing True --per_device_train_batch_size {PER_DEVICE_TRAIN_BATCH_SIZE} \
      --gradient_accumulation_steps {GRAD_ACCUMULATION} --learning_rate {LEARNING_RATE} --num_train_epochs 3 \
      --lr_scheduler_type cosine --warmup_steps {WARMUP_3E} --logging_steps 10 --logging_first_step True \
      --save_strategy epoch --report_to none \
      --remove_unused_columns False --dataloader_num_workers 2 --max_len {MAX_LEN} --seed {SEED}

In [ ]:
!python -u moss_infer.py --moss-repo {MOSS_REPO} --model-dir {MODEL_DIR} --adapter-dir {WINNER_DIR} --audio-dir data/dev --output {WINNER_PRED} --max-new-tokens {MAX_NEW_TOKENS} --resume
!python -u eval_add2026.py {WINNER_PRED} --subset data/swift/dev.jsonl --out-json {WINNER_EVAL} --bert-batch-size 32 --bert-device cuda:0

In [ ]:
summary = {"model": MODEL_ID, "max_len": MAX_LEN, "profile_100_steps": json.loads(PROFILE_JSON.read_text(encoding="utf-8")), "one_epoch_experiments": comparison, "winner_rank": WINNER_RANK, "winner_three_epochs": compact_result(WINNER_RANK, WINNER_EVAL), "adapter_dir": WINNER_DIR.as_posix(), "predictions": WINNER_PRED.as_posix()}
summary_path = DRIVE_ROOT / "experiment_summary.json"
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2)); print(f"[saved] {summary_path}")